# Sentiment analysis 

## 1. Textblob-FR

Documentation: https://textblob.readthedocs.io/en/dev/

### Imports

In [2]:
from textblob import Blobber
from textblob_fr import PatternTagger, PatternAnalyzer
import os 
import nltk 
nltk.download('punkt_tab') 
from nltk.tokenize import sent_tokenize

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Mirei\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


### Création d'une fonction `get_sentiment`

In [3]:
tb = Blobber(pos_tagger=PatternTagger(), analyzer=PatternAnalyzer())

def get_sentiment(input_text):
    blob = tb(input_text)
    polarity, subjectivity = blob.sentiment
    polarity_perc = f"{100*abs(polarity):.0f}"
    subjectivity_perc = f"{100*subjectivity:.0f}"
    if polarity > 0:
        polarity_str = f"{polarity_perc}% positive"
    elif polarity < 0:
        polarity_str = f"{polarity_perc}% negative"
    else:
        polarity_str = "neutral"
    if subjectivity > 0:
        subjectivity_str = f"{subjectivity_perc}% subjective"
    else:
        subjectivity_str = "perfectly objective"
    print(f"This text is {polarity_str} and {subjectivity_str}.")

### Analyser le sentiment d'une phrase

In [4]:
get_sentiment("Ce journal est vraiment super intéressant.")

This text is 65% positive and 75% subjective.


In [5]:
get_sentiment("Cette phrase est négative et je ne suis pas content !")

This text is 41% negative and 60% subjective.


## 2. Utilisation de transformers

Documentation: https://github.com/TheophileBlard/french-sentiment-analysis-with-bert

**!!** Si le code ne tourne pas sur votre machine, vous pouvez le tester directement sur Google Colab en utilisant [ce lien](https://colab.research.google.com/github/TheophileBlard/french-sentiment-analysis-with-bert/blob/master/colab/french_sentiment_analysis_with_bert.ipynb) **!!**

Le modèle peut également être testé en ligne sur [HuggingFace](https://huggingface.co/tblard/tf-allocine)

Note : `transformers` v5 a supprimé le support TensorFlow. On utilise donc ici `philschmid/pt-tblard-tf-allocine`, un portage PyTorch du même modèle, avec les classes `AutoModelForSequenceClassification` (au lieu de `TFAutoModelForSequenceClassification`).

### Installation des librairies et imports

In [6]:
!uv pip install sentencepiece
!uv pip install transformers>=5.19.0
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline

Using Python 3.13.15 environment at: C:\Users\Mirei\tac\.venv
Checked 1 package in 31ms
Using Python 3.13.15 environment at: C:\Users\Mirei\tac\.venv
Checked 1 package in 21ms
c:\Users\Mirei\tac\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Chargement du modèle

In [7]:
tokenizer = AutoTokenizer.from_pretrained("philschmid/pt-tblard-tf-allocine")
model = AutoModelForSequenceClassification.from_pretrained("philschmid/pt-tblard-tf-allocine")

sentiment_analyser = pipeline('sentiment-analysis', model=model, tokenizer=tokenizer)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 12519.38it/s]


### Analyser le sentiment d'une phrase

In [8]:
sentiment_analyser("Ce journal est vraiment super intéressant.")

[{'label': 'POSITIVE', 'score': 0.985571026802063}]

In [9]:
sentiment_analyser("Cette phrase est négative et je ne suis pas content !")

[{'label': 'NEGATIVE', 'score': 0.9271845817565918}]

## Analyser 10 phrases de l'année 1914

In [10]:
# Lister les fichiers de l'année choisie 
year = 1914
data_path = "../data/txt"
files_year = sorted([f for f in os.listdir(data_path) if f.split("_")[2].split("-")[0] == str(year)]) 
len(files_year)

100

In [11]:
# Fonction pour chercher des phrases contenant un mot (pour choisir les 10 phrases)
def chercher_phrases(mot, maximum=5):   
    trouvees = 0
    for f in files_year:
        text = open(os.path.join(data_path, f), 'r', encoding='utf-8').read()
        for sentence in sent_tokenize(text, language="french"):
            if mot in sentence and trouvees < maximum:
                print(f"{f.split('_')[2]} : {sentence.replace(chr(10), ' ')}")
                print()
                trouvees += 1

In [12]:
# Chercher des phrases candidates par thème
for mot in ["loi scolaire", "Woeste", "Wilmart", "Serajevo", "soldats", "Prosper", "Pégoud", "Agence Rossel"]: 
    print(f"===== {mot} =====") 
    chercher_phrases(mot)

===== loi scolaire =====
1914-01-15 : B est regrettable que la loi scolaire soit là.

1914-01-15 : ta loi scolaire ' ta Chambre reprend ta discussion du projet 'de toi scolaire, en son article 15 ' if.

1914-01-15 : La loi scolaire (suite) 8J.

1914-04-04 : La loi scolaire ‘ Les.

1914-04-04 : — Je n’ai pas voté contre pour ne pas priver la droite d’une unité au moment où commence le débat sur la loi scolaire.

===== Woeste =====
1914-01-21 : M. Woeste.

1914-01-21 : • < M. Woeste.

1914-01-21 : M. Woeste.

1914-02-06 : S'adressant à M. Woeste, le sénateur de Bruxelles s’écriait : « Vous devriez vous joindre à moi.

1914-02-06 : M° Woeste, pour saluer ces jeunes milices et vous réjouir de leur courage. ’

===== Wilmart =====
1914-05-21 : ; Wilmart, id.

1914-06-24 : —• Vous ayez porté en trop, au débit de Wilmart, des titres de la première émis sion. ‘

1914-06-24 : • : L’expert prétend que Wilmart aurait remis en oireulation.des titres sortis, au tirage...La défense--.protesté' viveme

In [13]:
# Les 10 phrases sélectionnées (texte tel quel, avec les erreurs d'OCR) 
phrases = [ "B est regrettable que la loi scolaire soit là.",
            "— Je n’ai pas voté contre pour ne pas priver la droite d’une unité au moment où commence le débat sur la loi scolaire.", 
            "S'adressant à M. Woeste, le sénateur de Bruxelles s’écriait : « Vous devriez vous joindre à moi.", 
            "L’expert prétend que Wilmart aurait remis en oireulation.des titres sortis, au tirage...La défense--.protesté' vivement, mais cela n’éinèut pas le témoin, qui persiste dans ses allégations.", 
            "Sir Edward Grey a exprimé tout d’abord ses condoléances les 'plus vives nu sujet de l'as *asinat de Serajevo, Parlant ensuite des intérêts britanniques dans le golfe Persiqhe.", 
            "Les corps de i’archiduc et de la duchesse transférés à Vienne Serajevo, 29 Juin.", 
            "on a en effet une vue d’ensemble que ne bornent ni les arbres, ni les toits du château... Lors des manœuvres d’infanterie, l’an dernier, le coup d’œil était vraiment pittoresque: les soldats campaient là, au-dessous et leur activité mettait une note vivante dans noire solitude, tandis qu’elle réconciliait nos paysans avec l’idée de donner leurs fils à la patrie... Par les temps clairs, on voit la mer...", 
            "Prosper avait sa redingote et son chapeau haut-de-forme; elle sa robe de noces, copiée sur un audacieux modèle parisien.", 
            "Aviation FCGOUD, FEUILLE MORTE ï Au cours d’une de ses récentes séances,TAoadémie des Sports de Paris a décerné‘b Pégoud le grand- prix Henri Deutsch (de la Meürthe), d’une importance de 10,000 francs.", 
            "Ceux qui ont ainsi oritiqué la décision d’une assemblée qui a étudié, de très près les titres d’une quarantaine d’athlètes n’ont pas été sans doute suffisamment informés des raisons pour lesquelles ce grand-prix a été attribué b Pégoud.", 
] 
len(phrases)

10

In [14]:
# Analyser le sentiment de chaque phrase avec TextBlob 
for phrase in phrases: 
    print(phrase) 
    get_sentiment(phrase)
    print()

B est regrettable que la loi scolaire soit là.
This text is 10% positive and perfectly objective.

— Je n’ai pas voté contre pour ne pas priver la droite d’une unité au moment où commence le débat sur la loi scolaire.
This text is 14% positive and perfectly objective.

S'adressant à M. Woeste, le sénateur de Bruxelles s’écriait : « Vous devriez vous joindre à moi.
This text is neutral and perfectly objective.

L’expert prétend que Wilmart aurait remis en oireulation.des titres sortis, au tirage...La défense--.protesté' vivement, mais cela n’éinèut pas le témoin, qui persiste dans ses allégations.
This text is neutral and perfectly objective.

Sir Edward Grey a exprimé tout d’abord ses condoléances les 'plus vives nu sujet de l'as *asinat de Serajevo, Parlant ensuite des intérêts britanniques dans le golfe Persiqhe.
This text is 20% positive and 19% subjective.

Les corps de i’archiduc et de la duchesse transférés à Vienne Serajevo, 29 Juin.
This text is neutral and perfectly objective.

### Vérification : effet de la correction de l'OCR

In [15]:
# Vérification : la correction manuelle de l'OCR change-t-elle le résultat ? 
verification = [ 
    "B est regrettable que la loi scolaire soit là.", 
    "Il est regrettable que la loi scolaire soit là.", 
    "Sir Edward Grey a exprimé tout d’abord ses condoléances les 'plus vives nu sujet de l'as *asinat de Serajevo, Parlant ensuite des intérêts britanniques dans le golfe Persiqhe.", 
    "Sir Edward Grey a exprimé tout d’abord ses condoléances les plus vives au sujet de l'assassinat de Serajevo. Parlant ensuite des intérêts britanniques dans le golfe Persique.", 
    "C'est un horrible assassinat.", 
    "Il est triste.", 
] 
for phrase in verification: 
    print(phrase) 
    get_sentiment(phrase) 
    print()

B est regrettable que la loi scolaire soit là.
This text is 10% positive and perfectly objective.

Il est regrettable que la loi scolaire soit là.
This text is 10% positive and perfectly objective.

Sir Edward Grey a exprimé tout d’abord ses condoléances les 'plus vives nu sujet de l'as *asinat de Serajevo, Parlant ensuite des intérêts britanniques dans le golfe Persiqhe.
This text is 20% positive and 19% subjective.

Sir Edward Grey a exprimé tout d’abord ses condoléances les plus vives au sujet de l'assassinat de Serajevo. Parlant ensuite des intérêts britanniques dans le golfe Persique.
This text is 21% positive and 18% subjective.

C'est un horrible assassinat.
This text is 100% negative and 100% subjective.

Il est triste.
This text is 30% negative and 60% subjective.



### Analyser les 10 phrases avec le modèle BERT

In [19]:
# Analyser les 10 phrases avec le modèle BERT 
for phrase in phrases: 
    print(phrase) 
    print(sentiment_analyser(phrase)) 
    print()

B est regrettable que la loi scolaire soit là.
[{'label': 'POSITIVE', 'score': 0.5373594760894775}]

— Je n’ai pas voté contre pour ne pas priver la droite d’une unité au moment où commence le débat sur la loi scolaire.
[{'label': 'NEGATIVE', 'score': 0.5628952383995056}]

S'adressant à M. Woeste, le sénateur de Bruxelles s’écriait : « Vous devriez vous joindre à moi.
[{'label': 'POSITIVE', 'score': 0.5274965763092041}]

L’expert prétend que Wilmart aurait remis en oireulation.des titres sortis, au tirage...La défense--.protesté' vivement, mais cela n’éinèut pas le témoin, qui persiste dans ses allégations.
[{'label': 'NEGATIVE', 'score': 0.6024200916290283}]

Sir Edward Grey a exprimé tout d’abord ses condoléances les 'plus vives nu sujet de l'as *asinat de Serajevo, Parlant ensuite des intérêts britanniques dans le golfe Persiqhe.
[{'label': 'POSITIVE', 'score': 0.6456422209739685}]

Les corps de i’archiduc et de la duchesse transférés à Vienne Serajevo, 29 Juin.
[{'label': 'POSITIVE